In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import jieba
import re
import numpy as np
import copy
from transformers import BertTokenizer

# ========== basic configuration ==========
fix_length = 64
label_list = ['negative', 'positive']
vocab_size = 21128
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

# ========== tokenize and preprocess ==========
def jieba_tokenize(x, word_freq=None, stop_words=None):
    x = re.sub('[^\u4e00-\u9fa5]', "", str(x))
    tokens = jieba.lcut(x)
    if word_freq:
        tokens = [
            t for t in tokens 
            if word_freq[t] < 10000 and (stop_words is None or t not in stop_words) and len(t) > 1
        ]
    else:
        tokens = [t for t in tokens if stop_words is None or t not in stop_words and len(t) > 1]
    return tokens

def preprocess(text):
    return " ".join(jieba_tokenize(text))

# ========== model structure ==========
# -- Transformer --

class Transformer(nn.Module):
    def __init__(self, vocab_size=21128, seq_len=fix_length, n_class=2,
                 embed_dim=256, dim_model=256, dropout=0.2, num_head=8, hidden=512, num_encoder=4):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = PositionalEncoding(embed_dim, seq_len, dropout)
        self.encoder = Encoder(dim_model, num_head, hidden, dropout)
        self.encoders = nn.ModuleList([copy.deepcopy(self.encoder) for _ in range(num_encoder)])
        self.fc = nn.Linear(seq_len * dim_model, n_class)

    def forward(self, input_ids):
        out = self.embedding(input_ids)
        out = self.position_embedding(out)
        for encoder in self.encoders:
            out = encoder(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, embed, seq_len, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(seq_len, embed)
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed, 2) * -(np.log(10000.0) / embed))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :].to(x.device)
        return self.dropout(x)

class Encoder(nn.Module):
    def __init__(self, dim_model, num_head, hidden, dropout):
        super(Encoder, self).__init__()
        self.attn = MultiHeadAttention(dim_model, num_head, dropout)
        self.ff = PositionWiseFeedForward(dim_model, hidden, dropout)

    def forward(self, x):
        x = self.attn(x)
        x = self.ff(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim_model, num_head, dropout=0.1):
        super().__init__()
        assert dim_model % num_head == 0
        self.dim_head = dim_model // num_head
        self.num_head = num_head
        self.q_linear = nn.Linear(dim_model, dim_model)
        self.k_linear = nn.Linear(dim_model, dim_model)
        self.v_linear = nn.Linear(dim_model, dim_model)
        self.fc = nn.Linear(dim_model, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        batch_size, seq_len, dim = x.size()
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.dim_head)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        out = self.fc(context)
        out = self.dropout(out)
        return self.norm(out + x)

class PositionWiseFeedForward(nn.Module):
    def __init__(self, dim_model, hidden, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim_model, hidden)
        self.fc2 = nn.Linear(hidden, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        out = self.fc2(torch.relu(self.fc1(x)))
        out = self.dropout(out)
        return self.norm(out + x)

# -- TextCNN --
class TextCNN(nn.Module):
    def __init__(self, vocab_size=21128, embed_dim=128, class_num=2, kernel_nums=128, kernel_sizes=[3,4,5], dropout=0.7):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([nn.Conv2d(1, kernel_nums, (k, embed_dim)) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(kernel_sizes) * kernel_nums, class_num)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = x.unsqueeze(1)
        x = [F.relu(conv(x)).squeeze(3) for conv in self.convs]
        x = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in x]
        x = torch.cat(x, 1)
        x = self.dropout(x)
        return self.fc(x)

# -- TextRCNN --
class TextRCNN(nn.Module):
    def __init__(self, vocab_size=21128, n_class=2, embed_dim=256, rnn_hidden=384, dropout=0.1):
        super(TextRCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, rnn_hidden, 2, bidirectional=True, batch_first=True, dropout=dropout)
        self.maxpool = nn.MaxPool1d(fix_length)
        self.fc = nn.Linear(embed_dim + 2 * rnn_hidden, n_class)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        x = torch.cat([x, output], dim=2)
        x = F.relu(x)
        x = x.permute(0, 2, 1)
        x = self.maxpool(x).squeeze()
        return self.fc(x)

# -- TextRNN --
class TextRNN(nn.Module):
    def __init__(self, vocab_size=21128, embed_dim=256, hidden_size=256, n_class=2, num_layers=2, dropout=0.2):
        super(TextRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size * 2, n_class)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        output, _ = self.lstm(x)
        out = output[:, -1, :]
        return self.fc(out)

# -- TextRNN + Attention --
class TextRNN_Attention(nn.Module):
    def __init__(self, vocab_size=21128, embed_dim=256, hidden_size=384, n_class=2, num_layers=3, dropout=0.2):
        super(TextRNN_Attention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers, bidirectional=True, batch_first=True, dropout=dropout)
        self.tanh = nn.Tanh()
        self.attention_weight = nn.Parameter(torch.zeros(hidden_size * 2))
        self.fc = nn.Linear(hidden_size * 2, n_class)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        output, _ = self.lstm(x)
        M = self.tanh(output)
        alpha = F.softmax(torch.matmul(M, self.attention_weight), dim=1).unsqueeze(-1)
        out = output * alpha
        out = torch.sum(out, dim=1)
        out = F.relu(out)
        return self.fc(out)

# ======= Generalized prediction functions ========
def classify(text, model, tokenizer, word_freq=None, stop_words=None, device='cpu'):
    model.to(device)
    model.eval()
    text = " ".join(jieba_tokenize(text, word_freq=word_freq, stop_words=stop_words))
    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=fix_length,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)

    with torch.no_grad():
        output = model(input_ids)
        pred = torch.argmax(output, dim=1).item()
        return label_list[pred]

In [11]:
import pandas as pd
from collections import Counter
import re
import jieba

# Load training dataset text for word frequency
train_df = pd.read_csv("/kaggle/input/douban-mgh/train_pinggu.csv")

# Word frequency
word_freq = Counter()
for text in train_df['text']:
    x = re.sub('[^\u4e00-\u9fa5]', "", str(text))
    word_freq.update(jieba.lcut(x))

# Load model parameters
model = Transformer(n_class=2)
model.load_state_dict(torch.load('/kaggle/input/notebook065e548398/Transformer_model.pt', map_location='cpu'))

# input a sentence to make predictions
text = "怪不得主旋律电影烂，人物没一个立起来，电影里一点感情都没有。谁死了我都觉得好蠢，丝毫没有感伤的感觉。配乐也是，有时候没有背景音乐比有更好的道理难道整个团队没一个人懂?其实不止一星，但鉴于现在居然有7.8分，我只能给一星了"

result = classify(text, model, tokenizer, word_freq=word_freq)
print("classify output：", result)

分类结果： negative


<ipython-input-11-6edbd9595e72>:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/input/notebook065e548398/Transformer_model.pt', m